In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib
chunks = pd.read_csv('../data/accepted_2007_to_2018Q4.csv', chunksize=100000, parse_dates=['issue_d'], low_memory=False)
df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_78038/387980389.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])
/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_78038/387980389.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])
/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_78038/387980389.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please

In [6]:
import os

In [7]:
os.makedirs("data_splits", exist_ok=True)

def clean_series(s: pd.Series) -> pd.Series:
    if s.dtype != "object":
        return s
    def clean_val(v):
        if pd.isna(v):
            return None
        if isinstance(v, (bytes, bytearray)):
            return v.decode("utf-8", errors="replace")
        return str(v)
    return s.map(clean_val)

In [ ]:
df_filtered["quarter"] = df_filtered["issue_d"].dt.to_period("Q")

for q, grp in df_filtered.groupby("quarter"):
    for col in grp.select_dtypes(include=["object"]).columns:
        grp[col] = clean_series(grp[col])
    fname = f"data_splits/data_{q}.parquet"
    grp.to_parquet(fname, index=False, compression="snappy")